In [ ]:
import os, gc, json, glob, time
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd
from tensorflow.keras import mixed_precision

# REPRODUCIBILITY PROTOCOL 
np.random.seed(42)
tf.random.set_seed(42)
os.environ['PYTHONHASHSEED'] = '42'

# COMPUTATIONAL POLICY 
mixed_precision.set_global_policy('mixed_float16')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

def setup_working_directory():
    for folder in ['./models', './logs', './results']:
        os.makedirs(folder, exist_ok=True)
    print("STATUS: Directory structure established at /models, /logs, and /results")

# HARDWARE ACCELERATION SETUP 
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
        print(f"HARDWARE: P100 GPU identified and initialized.")
        print(f"PRECISION POLICY: mixed_float16 enabled for Tensor Core acceleration.")
    except RuntimeError as e:
        print(f"CONFIGURATION ERROR: GPU initialization failed with: {e}")
else:
    print("WARNING: No GPU hardware detected. Utilizing CPU fallback.")

setup_working_directory()
gc.collect()


print("INITIALIZATION COMPLETE: RESEARCH ENVIRONMENT SECURED")


In [ ]:
def load_npz_file(file_path):
    path = file_path.numpy().decode('utf-8')
    data = np.load(path)
    image = data['image'].astype(np.float32)
    image = np.clip(image, 0.0, 1.0)
    mask = data['mask'].astype(np.float32)
    if mask.max() > 1.0:
        mask = mask / 255.0
    mask = (mask > 0.5).astype(np.float32)
    if len(mask.shape) == 2:
        mask = np.expand_dims(mask, axis=-1)
    return image, mask

def tf_parse_func(file_path):
    image, mask = tf.py_function(load_npz_file, [file_path], [tf.float32, tf.float32])
    image = tf.ensure_shape(image, [256, 256, 3])
    mask = tf.ensure_shape(mask, [256, 256, 1])
    return image, mask

def augment_data(image, mask):
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_left_right(image)
        mask = tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_up_down(image)
        mask = tf.image.flip_up_down(mask)
    k = tf.random.uniform((), maxval=4, dtype=tf.int32)
    image = tf.image.rot90(image, k)
    mask = tf.image.rot90(mask, k)
    return image, mask

def get_dataset(data_dir, batch_size=32, shuffle=True, augment=False):
    file_list = glob.glob(os.path.join(data_dir, "*.npz"))
    if not file_list:
        raise ValueError(f"Operational Failure: No .npz files identified in {data_dir}")
    ds = tf.data.Dataset.from_tensor_slices(file_list)
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(file_list), 1000))
    ds = ds.map(tf_parse_func, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(augment_data, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

DATASET_BASE = "/kaggle/input/deforestation-data-3/deforestation_data_final"
train_ds = get_dataset(os.path.join(DATASET_BASE, "train"), batch_size=32, shuffle=True, augment=True)
val_ds = get_dataset(os.path.join(DATASET_BASE, "val"), batch_size=32, shuffle=False, augment=False)
test_ds = get_dataset(os.path.join(DATASET_BASE, "test"), batch_size=32, shuffle=False, augment=False)

for img_batch, mask_batch in train_ds.take(1):
    positive_ratio = np.sum(mask_batch.numpy() > 0.5) / mask_batch.numpy().size
    print(f"VALIDATION: Sample Batch Intensity Range: [{img_batch.numpy().min():.2f}, {img_batch.numpy().max():.2f}]")
    print(f"VALIDATION: Positive Class Density: {positive_ratio*100:.2f}%")

In [ ]:
from tensorflow.keras import layers, models

# 1. KEPT: Your original Residual Block
def residual_convolution_block(x, filters, dropout=0.0):
    shortcut = layers.Conv2D(filters, 1, padding='same', kernel_initializer='he_normal')(x)
    shortcut = layers.BatchNormalization()(shortcut)
    x = layers.Conv2D(filters, 3, padding='same', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, 3, padding='same', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([shortcut, x])
    x = layers.Activation('relu')(x)
    if dropout > 0:
        x = layers.Dropout(dropout)(x)
    return x

# 2. KEPT: Your original Proposed ResUNet
def build_resunet_stable(input_shape=(256, 256, 3), dropout_rate=0.3):
    inputs = layers.Input(input_shape, name='input')
    c1 = residual_convolution_block(inputs, 64, dropout=0)
    p1 = layers.MaxPooling2D(2)(c1)
    c2 = residual_convolution_block(p1, 128, dropout=dropout_rate)
    p2 = layers.MaxPooling2D(2)(c2)
    c3 = residual_convolution_block(p2, 256, dropout=dropout_rate)
    p3 = layers.MaxPooling2D(2)(c3)
    c4 = residual_convolution_block(p3, 512, dropout=dropout_rate)
    p4 = layers.MaxPooling2D(2)(c4)
    bn = residual_convolution_block(p4, 1024, dropout=dropout_rate)
    u4 = layers.Conv2DTranspose(512, 2, strides=2, padding='same')(bn)
    u4 = layers.Concatenate()([u4, c4])
    uc4 = residual_convolution_block(u4, 512, dropout=dropout_rate)
    u3 = layers.Conv2DTranspose(256, 2, strides=2, padding='same')(uc4)
    u3 = layers.Concatenate()([u3, c3])
    uc3 = residual_convolution_block(u3, 256, dropout=dropout_rate)
    u2 = layers.Conv2DTranspose(128, 2, strides=2, padding='same')(uc3)
    u2 = layers.Concatenate()([u2, c2])
    uc2 = residual_convolution_block(u2, 128, dropout=dropout_rate)
    u1 = layers.Conv2DTranspose(64, 2, strides=2, padding='same')(uc2)
    u1 = layers.Concatenate()([u1, c1])
    uc1 = residual_convolution_block(u1, 64, dropout=0)
    outputs = layers.Conv2D(1, 1, activation='sigmoid', dtype='float32', name='output')(uc1)
    return models.Model(inputs, outputs, name='ResUNet_Stable')

# 3. KEPT: Your original FCN Baseline
def build_fcn_baseline(input_shape=(256, 256, 3)):
    inputs = layers.Input(input_shape, name='input')
    x = layers.Conv2D(64, 3, activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(128, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(256, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.UpSampling2D(2)(x)
    x = layers.Conv2D(128, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.UpSampling2D(2)(x)
    x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    outputs = layers.Conv2D(1, 1, activation='sigmoid', dtype='float32', name='output')(x)
    return models.Model(inputs, outputs, name='FCN_Baseline')

# 4. NEW ADDITION: Standard U-Net Baseline (Satisfies Section 4.3 "2 Baseline Methods")

def build_simple_unet_baseline(input_shape=(256, 256, 3)):
    inputs = layers.Input(input_shape)
    # Simple Encoder
    c1 = layers.Conv2D(32, 3, activation='relu', padding='same')(inputs)
    p1 = layers.MaxPooling2D(2)(c1)
    c2 = layers.Conv2D(64, 3, activation='relu', padding='same')(p1)
    p2 = layers.MaxPooling2D(2)(c2)
    # Bottleneck
    b = layers.Conv2D(128, 3, activation='relu', padding='same')(p2)
    # Simple Decoder
    u1 = layers.UpSampling2D(2)(b)
    c3 = layers.Conv2D(64, 3, activation='relu', padding='same')(u1)
    u2 = layers.UpSampling2D(2)(c3)
    c4 = layers.Conv2D(32, 3, activation='relu', padding='same')(u2)
    outputs = layers.Conv2D(1, 1, activation='sigmoid', dtype='float32', name='output')(c4)
    return models.Model(inputs, outputs, name='Standard_UNet_Baseline')

In [ ]:
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(tf.cast(y_true, tf.float32), [-1])
    y_pred_f = tf.reshape(tf.cast(y_pred, tf.float32), [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

# --- NEW: Weighted Loss to fix False Negatives ---
def weighted_combined_loss_stable(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    y_pred = tf.clip_by_value(y_pred, 1e-6, 1.0 - 1e-6)
    
    # pos_weight=2.0 forces the model to care 2x more about deforested pixels
    bce = tf.nn.weighted_cross_entropy_with_logits(y_true, y_pred, pos_weight=2.0)
    dice = 1.0 - dice_coefficient(y_true, y_pred)
    return tf.reduce_mean(bce) + dice

def iou_metric(y_true, y_pred, threshold=0.5, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred > threshold, tf.float32)
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

def evaluate_model(model, model_name, test_ds):
    print(f"STATUS: Running inference on Test Set for {model_name}...")
    results = model.evaluate(test_ds, verbose=1)
    return dict(zip(model.metrics_names, results))

def save_and_display_plots(history, model_name):
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    metrics = [('loss', 'Loss Profile'), ('accuracy', 'Accuracy Profile'), 
               ('dice_coefficient', 'Dice Coefficient'), ('iou_metric', 'Jaccard Index (IoU)')]
    for i, (key, title) in enumerate(metrics):
        ax = axes[i // 2, i % 2]
        if key in history.history:
            ax.plot(history.history[key], label='Training Phase')
            ax.plot(history.history[f'val_{key}'], label='Validation Phase')
            ax.set_title(title); ax.legend(); ax.grid(True)
    plt.tight_layout()
    plt.savefig(f'./results/{model_name}_convergence.png', dpi=300)
    plt.show()

# Updated to use Weighted Loss
def compile_and_train_stable(model, model_name, train_ds, val_ds, epochs=50, lr=1e-5):
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr, clipnorm=1.0)
    model.compile(optimizer=optimizer, loss=weighted_combined_loss_stable, 
                  metrics=['accuracy', dice_coefficient, iou_metric, 
                           tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')])
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(f'./models/{model_name}_best.keras', monitor='val_dice_coefficient', mode='max', save_best_only=True, verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor='val_dice_coefficient', patience=10, mode='max', restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1),
        tf.keras.callbacks.TerminateOnNaN()
    ]
    history = model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=callbacks, verbose=1)
    save_and_display_plots(history, model_name)
    return history

In [ ]:
import shutil

# KEPT: Your original logic for identifying high-deforestation samples
def filter_deforestation_samples(data_dir, threshold=0.05):
    all_files = glob.glob(os.path.join(data_dir, "*.npz"))
    filtered_files = []
    print(f"Scanning {len(all_files)} files for heavy deforestation (>{threshold*100}%)...")
    for f in all_files:
        data = np.load(f)
        if np.mean(data['mask'] > 0) >= threshold:
            filtered_files.append(f)
    print(f"Filter complete: Found {len(filtered_files)} 'Hard' samples.")
    return filtered_files

# KEPT: Your original academic table generator
def generate_academic_comparison_table(results_list):
    os.makedirs('./results', exist_ok=True)
    table_path = './results/comparison_table.txt'
    with open(table_path, 'w') as f:
        f.write("RESEARCH COMPARISON TABLE: DEFORESTATION SEGMENTATION PERFORMANCE\n" + "="*90 + "\n")
        f.write(f"{'Model Identity':<25} {'Accuracy':<10} {'Dice':<10} {'IoU':<10} {'Precision':<10}\n" + "-"*90 + "\n")
        for res in results_list:
            f.write(f"{res['model_name']:<25} {res.get('accuracy', 0.0):<10.4f} {res.get('dice_coefficient', 0.0):<10.4f} "
                    f"{res.get('iou_metric', 0.0):<10.4f} {res.get('precision', 0.0):<10.4f}\n")
    print(f"STATUS: Table exported to {table_path}")

# UPDATED: Systematic pipeline now includes the 2nd Baseline required by the guide
def run_research_pipeline_systematic(train_ds, val_ds, test_ds, epochs=50):
    final_stats = []
    
    # --- UPDATED: 3 models instead of 2 to satisfy Section 4.3 ---
    experiments = [
        {"factory": build_resunet_stable, "name": "ResUNet_Proposed"},
        {"factory": build_fcn_baseline, "name": "FCN_Baseline"},
        {"factory": build_simple_unet_baseline, "name": "Standard_UNet_Baseline"} 
    ]
    
    for i, exp in enumerate(experiments):
        model_name = exp["name"]
        tf.random.set_seed(42 + i); np.random.seed(42 + i)
        step_start = time.time()
        try:
            # --- STAGE 1: BASE WEIGHTED TRAINING ---
            model = exp["factory"]()
            history = compile_and_train_stable(model, model_name, train_ds, val_ds, epochs=epochs)
            
            # --- STAGE 2: FINE-TUNING (KEPT: Specific to your Proposed ResUNet) ---
            if "ResUNet" in model_name:
                print(f"\nSTATUS: Initiating Fine-Tuning for {model_name}...")
                hard_files = filter_deforestation_samples(os.path.join(DATASET_BASE, "train"), threshold=0.05)
                fine_tune_ds = tf.data.Dataset.from_tensor_slices(hard_files).map(tf_parse_func).map(augment_data).batch(16).prefetch(2)
                # Re-train with ultra-low learning rate
                history_fine = compile_and_train_stable(model, f"{model_name}_Tuned", fine_tune_ds, val_ds, epochs=15, lr=1e-6)

            # Evaluation on Test Set
            metrics = evaluate_model(model, model_name, test_ds)
            result_entry = {'model_name': model_name, 'parameters_m': round(model.count_params() / 1e6, 2),
                            'training_time_minutes': round((time.time() - step_start) / 60, 2), **metrics}
            final_stats.append(result_entry)
            
            # Memory Management
            tf.keras.backend.clear_session(); del model; gc.collect()
            
        except Exception as e: 
            print(f"ERROR: {model_name} failed with: {e}")
            continue
    
    # Save summary data
    with open('./results/comparative_research_summary.json', 'w') as f: 
        json.dump(final_stats, f, indent=4)
        
    generate_academic_comparison_table(final_stats)
    
    # Artifact Archiving
    try: 
        shutil.make_archive('Deforestation_Research_Final_Artifacts', 'zip', './results')
    except: 
        pass
        
    return final_stats

# Execute Pipeline
pipeline_results = run_research_pipeline_systematic(train_ds, val_ds, test_ds, epochs=50)

In [ ]:
def generate_visual_evidence(model_path, dataset, model_name, num_samples=3):
    """
    Generates comparison maps for Section 3.6.2. 
    Shows Satellite Input vs. Ground Truth vs. Predicted Map.
    """
    if not os.path.exists(model_path):
        print(f"SKIPPING: {model_name} (File not found at {model_path})")
        return

    print(f"Generating visual evidence for {model_name}...")
    
    custom_objs = {
        'weighted_combined_loss_stable': weighted_combined_loss_stable,
        'dice_coefficient': dice_coefficient,
        'iou_metric': iou_metric
    }
    
    # Load the specified model
    model = tf.keras.models.load_model(model_path, custom_objects=custom_objs, compile=False)
    
    # Use the same samples for all models to ensure a fair visual comparison
    for images, masks in dataset.take(1):
        predictions = model.predict(images, verbose=0)
        
        plt.figure(figsize=(18, num_samples * 4))
        for i in range(num_samples):
            # 1. Original Satellite Image Input
            plt.subplot(num_samples, 3, i*3 + 1)
            plt.imshow(images[i])
            plt.title(f"{model_name}\nSample {i+1}: Satellite")
            plt.axis('off')
            
            # 2. Ground Truth (Target)
            plt.subplot(num_samples, 3, i*3 + 2)
            plt.imshow(masks[i].numpy().squeeze(), cmap='gray')
            plt.title("Ground Truth (Actual)")
            plt.axis('off')
            
            # 3. Model's Predicted Segmentation Map
            plt.subplot(num_samples, 3, i*3 + 3)
            # Thresholding at 0.5 to show binary mask
            plt.imshow(predictions[i].squeeze() > 0.5, cmap='jet')
            plt.title("Model Prediction")
            plt.axis('off')
            
        plt.tight_layout()
        save_path = f'./results/{model_name}_visual_comparison.png'
        plt.savefig(save_path, dpi=300)
        print(f"SUCCESS: Plot saved to {save_path}")
        plt.show()

# --- EXECUTE FOR ALL 3 MODELS ---
# Note: Using the 'Tuned' version for the Proposed Model as requested
print("\n" + "="*30 + " GENERATING VISUAL COMPARISONS " + "="*30)

# 1. Main Proposed Model (The Fine-Tuned Version)
generate_visual_evidence('./models/ResUNet_Proposed_Tuned_best.keras', test_ds, "ResUNet_Proposed_Tuned")

# 2. Baseline 1
generate_visual_evidence('./models/FCN_Baseline_best.keras', test_ds, "FCN_Baseline")

# 3. Baseline 2
generate_visual_evidence('./models/Standard_UNet_Baseline_best.keras', test_ds, "Standard_UNet_Baseline")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

def detailed_error_analysis(model_path, dataset, model_name="Model"):
    """
    Performs a deep statistical evaluation of model classification errors.
    Generates a confusion matrix and classification report.
    """
    if not os.path.exists(model_path):
        print(f"SKIPPING ANALYSIS: {model_name} (File not found at {model_path})")
        return

    # 1. KEPT: Original custom object logic for model loading
    custom_objs = {
        'weighted_combined_loss_stable': weighted_combined_loss_stable,
        'dice_coefficient': dice_coefficient,
        'iou_metric': iou_metric
    }
    
    model = tf.keras.models.load_model(model_path, custom_objects=custom_objs, compile=False)
    
    all_masks = []
    all_preds = []
    
    print(f"\n" + "="*50)
    print(f"STATISTICAL ERROR ANALYSIS: {model_name}")
    print("="*50)
    
    # 2. KEPT: Processing 10 batches for statistical significance
    for images, masks in dataset.take(10): 
        preds = model.predict(images, verbose=0)
        all_masks.extend(masks.numpy().flatten())
        all_preds.extend((preds.flatten() > 0.5).astype(int))
        
    # 3. KEPT: Confusion Matrix generation with Seaborn
    cm = confusion_matrix(all_masks, all_preds)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Forest', 'Deforested'], 
                yticklabels=['Forest', 'Deforested'])
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(f'Confusion Matrix: {model_name}')
    
    # 4. KEPT: Saving the result for Section 5.1 compliance
    save_path = f'./results/{model_name}_confusion_matrix.png'
    plt.savefig(save_path, dpi=300) # Added high-res DPI
    plt.show()
    
    # 5. KEPT: Detailed Classification Report
    print(f"\nClassification Report for {model_name}:")
    report = classification_report(all_masks, all_preds, target_names=['Forest', 'Deforested'])
    print(report)
    
    # Save report to text file for easy inclusion in paper
    with open(f'./results/{model_name}_report.txt', 'w') as f:
        f.write(report)

# --- EXECUTE FOR ALL 3 MODELS ---
# Using the Tuned version of ResUNet to show the best possible results
detailed_error_analysis('./models/ResUNet_Proposed_Tuned_best.keras', test_ds, "ResUNet_Proposed_Tuned")
detailed_error_analysis('./models/FCN_Baseline_best.keras', test_ds, "FCN_Baseline")
detailed_error_analysis('./models/Standard_UNet_Baseline_best.keras', test_ds, "Standard_UNet_Baseline")

In [ ]:
from sklearn.metrics import roc_curve, auc, precision_recall_curve

def plot_research_curves(model_path, dataset, model_name="Model"):
    """
    Generates ROC and Precision-Recall curves as required by 
    Section 3.6.2 of the research guide.
    """
    if not os.path.exists(model_path): 
        print(f"Skipping curves for {model_name}: Model file not found at {model_path}")
        return

    # 1. KEPT: Original custom object logic for loading weighted models
    custom_objs = {
        'weighted_combined_loss_stable': weighted_combined_loss_stable,
        'dice_coefficient': dice_coefficient, 
        'iou_metric': iou_metric
    }
    
    print(f"STATUS: Extracting probability maps for {model_name} ROC/PR analysis...")
    model = tf.keras.models.load_model(model_path, custom_objects=custom_objs, compile=False)
    
    all_masks = []
    all_probs = []
    
    # 2. KEPT: Processing 20 batches for statistical sample size
    for images, masks in dataset.take(20):
        preds = model.predict(images, verbose=0)
        all_masks.extend(masks.numpy().flatten())
        all_probs.extend(preds.flatten())
        
    # --- Plotting ---
    plt.figure(figsize=(15, 6))
    
    # 3. KEPT: ROC Curve logic and AUC calculation
    fpr, tpr, _ = roc_curve(all_masks, all_probs)
    roc_auc = auc(fpr, tpr)
    
    plt.subplot(1, 2, 1)
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.4f}')
    plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
    plt.xlabel('False Positive Rate (FPR)')
    plt.ylabel('True Positive Rate (TPR/Recall)')
    plt.title(f'ROC Curve: {model_name}')
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    
    # 4. KEPT: Precision-Recall Curve logic
    precision, recall, _ = precision_recall_curve(all_masks, all_probs)
    
    plt.subplot(1, 2, 2)
    plt.plot(recall, precision, color='forestgreen', lw=2)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(f'Precision-Recall Curve: {model_name}')
    plt.grid(alpha=0.3)
    
    plt.tight_layout()
    
    # 5. KEPT: High-resolution (300 DPI) saving for Section 5.1 compliance
    save_path = f'./results/{model_name}_statistical_curves.png'
    plt.savefig(save_path, dpi=300)
    print(f"SUCCESS: Statistical curves saved to {save_path}")
    plt.show()

# --- EXECUTE FOR ALL 3 MODELS ---
# Using the 'Tuned' version for the ResUNet to show final research results
print("\n" + "="*30 + " GENERATING FINAL RESEARCH PLOTS " + "="*30)
plot_research_curves('./models/ResUNet_Proposed_Tuned_best.keras', test_ds, "ResUNet_Proposed_Tuned")
plot_research_curves('./models/FCN_Baseline_best.keras', test_ds, "FCN_Baseline")
plot_research_curves('./models/Standard_UNet_Baseline_best.keras', test_ds, "Standard_UNet_Baseline")